In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/dogs-vs-cats/test1.zip
/kaggle/input/competitions/dogs-vs-cats/train.zip
/kaggle/input/competitions/dogs-vs-cats/sampleSubmission.csv


In [2]:
import zipfile
import os

train_zip = '/kaggle/input/competitions/dogs-vs-cats/train.zip'

with zipfile.ZipFile(train_zip, 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working/train')

print("Training images extracted successfully!")

Training images extracted successfully!


In [3]:
train_path = '/kaggle/working/train/train'

print("Total images:", len(os.listdir(train_path)))
print("Sample filenames:", os.listdir(train_path)[:10])

Total images: 25000
Sample filenames: ['cat.5625.jpg', 'dog.4134.jpg', 'dog.4719.jpg', 'cat.11677.jpg', 'dog.12244.jpg', 'dog.7798.jpg', 'cat.74.jpg', 'dog.11764.jpg', 'cat.695.jpg', 'cat.10999.jpg']


In [4]:
import random
from PIL import Image
from tqdm import tqdm

random.seed(42)

files = os.listdir(train_path)

cats = [f for f in files if f.startswith('cat')]
dogs = [f for f in files if f.startswith('dog')]

selected = random.sample(cats, 1000) + random.sample(dogs, 1000)
random.shuffle(selected)

X = []
y = []

for filename in tqdm(selected):
    img = Image.open(os.path.join(train_path, filename))
    img = img.convert('L')
    img = img.resize((32, 32))

    X.append(np.array(img).flatten() / 255.0)

    if filename.startswith('cat'):
        y.append(0)
    else:
        y.append(1)

X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

100%|██████████| 2000/2000 [00:04<00:00, 430.47it/s]

Feature shape: (2000, 1024)
Label shape: (2000,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training images:", X_train.shape[0])
print("Testing images:", X_test.shape[0])

Training images: 1600
Testing images: 400


In [6]:
svm_model = SVC(kernel='rbf', C=1, gamma='scale')

svm_model.fit(X_train, y_train)

print("SVM model trained successfully!")

SVM model trained successfully!


In [7]:
y_pred = svm_model.predict(X_test)

print("Predictions completed!")
print("First 10 predictions:", y_pred[:10])

Predictions completed!
First 10 predictions: [1 0 1 1 1 1 0 0 0 1]


In [8]:
accuracy = accuracy_score(y_test, y_pred)

print("SVM Accuracy:", accuracy * 100, "%")

SVM Accuracy: 61.25000000000001 %


In [9]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Cat', 'Dog']
))

Confusion Matrix:
[[112  88]
 [ 67 133]]

Classification Report:
              precision    recall  f1-score   support

         Cat       0.63      0.56      0.59       200
         Dog       0.60      0.67      0.63       200

    accuracy                           0.61       400
   macro avg       0.61      0.61      0.61       400
weighted avg       0.61      0.61      0.61       400



In [10]:
from skimage.feature import hog

X_hog = []

for img in tqdm(X):
    img = img.reshape(32, 32)

    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    X_hog.append(features)

X_hog = np.array(X_hog)

print("HOG Feature Shape:", X_hog.shape)

100%|██████████| 2000/2000 [00:00<00:00, 2169.09it/s]

HOG Feature Shape: (2000, 324)


In [11]:
X_train_hog, X_test_hog, y_train_hog, y_test_hog = train_test_split(
    X_hog, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train_hog.shape)
print("Testing:", X_test_hog.shape)

Training: (1600, 324)
Testing: (400, 324)


In [12]:
svm_hog = SVC(kernel='rbf', C=10, gamma='scale')

svm_hog.fit(X_train_hog, y_train_hog)

print("Improved SVM model trained successfully!")

Improved SVM model trained successfully!


In [13]:
y_pred_hog = svm_hog.predict(X_test_hog)

accuracy_hog = accuracy_score(y_test_hog, y_pred_hog)

print("HOG + SVM Accuracy:", accuracy_hog * 100, "%")
print("Original SVM Accuracy: 59.5 %")

HOG + SVM Accuracy: 69.25 %
Original SVM Accuracy: 59.5 %


In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [1, 10, 50, 100],
    'gamma': ['scale', 0.001, 0.01, 0.1]
}

grid = GridSearchCV(
    SVC(kernel='rbf'),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_hog, y_train_hog)

print("Best Parameters:", grid.best_params_)
print("Best Cross-validation Accuracy:", grid.best_score_ * 100)

Best Parameters: {'C': 1, 'gamma': 0.1}
Best Cross-validation Accuracy: 71.1866967416433


In [15]:
best_svm = grid.best_estimator_

y_pred_tuned = best_svm.predict(X_test_hog)

tuned_accuracy = accuracy_score(y_test_hog, y_pred_tuned)

print("Final Tuned SVM Accuracy:", tuned_accuracy * 100, "%")
print("Previous HOG + SVM Accuracy:", 69.75, "%")

Final Tuned SVM Accuracy: 68.0 %
Previous HOG + SVM Accuracy: 69.75 %


In [16]:
cm = confusion_matrix(y_test_hog, y_pred_tuned)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test_hog,
    y_pred_tuned,
    target_names=['Cat', 'Dog']
))

Confusion Matrix:
[[133  67]
 [ 61 139]]

Classification Report:
              precision    recall  f1-score   support

         Cat       0.69      0.67      0.68       200
         Dog       0.67      0.69      0.68       200

    accuracy                           0.68       400
   macro avg       0.68      0.68      0.68       400
weighted avg       0.68      0.68      0.68       400

